# 00 – Setup & Erstmigration

Dieses Notebook richtet die Databricks-Umgebung ein und führt die **Erstmigration** durch.

**Einmalig ausführen** – danach für Updates `01_delta_update.ipynb` verwenden.

## Ablauf
1. Secrets prüfen (GitHub Codespaces)
2. Verbindung zu Databricks testen
3. Schema `main.polar` anlegen
4. Alle 5 Delta Tables erstellen
5. Erstes ZIP importieren
6. Tabellenübersicht ausgeben

---
> ⚠️ **Voraussetzung:** Alle 5 GitHub Codespaces Secrets müssen gesetzt sein.  
> Repository → Settings → Secrets and variables → Codespaces

## 1 – Imports & Secrets prüfen

In [ ]:
import os
import sys

# src/ zum Python-Pfad hinzufügen
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from db_loader import DatabricksLoader, secrets_pruefen

# ── Secrets prüfen ──────────────────────────────────────────
# Wirft EnvironmentError mit Schritt-für-Schritt-Anleitung
# wenn ein Secret fehlt
secrets_pruefen()

## 2 – Verbindung zu Databricks testen

In [ ]:
# Verbindungsparameter anzeigen (Token wird nicht ausgegeben)
print("Verbindungsparameter:")
print(f"  Host      : {os.environ.get('DATABRICKS_HOST')}")
print(f"  HTTP-Path : {os.environ.get('DATABRICKS_HTTP_PATH')}")
print(f"  Catalog   : {os.environ.get('DATABRICKS_CATALOG', 'main')}")
print(f"  Schema    : {os.environ.get('DATABRICKS_SCHEMA', 'polar')}")
print(f"  Token     : {'*' * 20} (ausgeblendet)")
print()

# Verbindung aufbauen und testen
db = DatabricksLoader()
ok = db.verbindung_testen()

if not ok:
    raise RuntimeError(
        "Verbindungstest fehlgeschlagen!\n"
        "→ SQL Warehouse im Databricks UI starten\n"
        "→ Token prüfen: Databricks UI → User Settings → Developer"
    )

## 3 – Schema anlegen

In [ ]:
catalog = os.environ.get('DATABRICKS_CATALOG', 'main')
schema  = os.environ.get('DATABRICKS_SCHEMA', 'polar')

print(f"Lege Schema an: {catalog}.{schema}")

db.abfrage(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

# Schema-Existenz bestätigen
df_schemas = db.abfrage(
    f"SHOW SCHEMAS IN {catalog} LIKE '{schema}'"
)
if not df_schemas.empty:
    print(f"✅ Schema '{catalog}.{schema}' ist bereit")
else:
    print(f"⚠️  Schema konnte nicht verifiziert werden – trotzdem fortfahren")

## 4 – Delta Tables erstellen

Alle Tabellen werden mit `CREATE TABLE IF NOT EXISTS` angelegt –
bestehende Daten bleiben erhalten.

In [ ]:
# ── Tabellen-Definitionen ────────────────────────────────────
tabellen_sql = {

    'activity': f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{schema}.activity (
            datum             DATE    NOT NULL COMMENT 'Datum des Aktivitätstages',
            schritte          BIGINT          COMMENT 'Tägliche Schrittanzahl',
            kalorien          DOUBLE          COMMENT 'Verbrauchte Kalorien (kcal)',
            schlaf_stunden    DOUBLE          COMMENT 'Schlafdauer in Stunden',
            schlaf_qualitaet  DOUBLE          COMMENT 'Schlafqualität 0-1',
            met_minuten       DOUBLE          COMMENT 'Metabolische Äquivalent-Minuten'
        )
        USING DELTA
        COMMENT 'Tägliche Aktivitätsdaten aus Polar activity_*.json'
        TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true')
    """,

    'training': f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{schema}.training (
            datum        DATE    NOT NULL COMMENT 'Datum der Trainingseinheit',
            sport        STRING  NOT NULL COMMENT 'Sportart (z.B. RUNNING, CYCLING)',
            dauer_min    DOUBLE           COMMENT 'Trainingsdauer in Minuten',
            hr_avg       DOUBLE           COMMENT 'Durchschnittliche Herzfrequenz (bpm)',
            hr_max       DOUBLE           COMMENT 'Maximale Herzfrequenz (bpm)',
            distanz_km   DOUBLE           COMMENT 'Zurückgelegte Distanz in km',
            kalorien     DOUBLE           COMMENT 'Verbrannte Kalorien (kcal)',
            wochentag    STRING           COMMENT 'Wochentag (Monday..Sunday)',
            jahr         INT              COMMENT 'Jahr der Trainingseinheit'
        )
        USING DELTA
        COMMENT 'Trainingseinheiten aus Polar training_*.json'
        TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true')
    """,

    'heartrate': f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{schema}.heartrate (
            datum         DATE    NOT NULL COMMENT 'Messdatum',
            hr_ruhepuls   DOUBLE           COMMENT 'Ruhepuls: 5. Perzentil aller Tages-HR-Werte',
            hr_mean       DOUBLE           COMMENT 'Mittlere Herzfrequenz des Tages',
            hr_max        INT              COMMENT 'Maximale Herzfrequenz des Tages',
            wochentag_nr  INT              COMMENT 'Wochentag 0=Montag, 6=Sonntag',
            monat         INT              COMMENT 'Monat 1-12'
        )
        USING DELTA
        COMMENT 'Tägliche HR-Aggregate aus Polar 247ohr_*.json'
        TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true')
    """,

    'hrv': f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{schema}.hrv (
            datum           DATE    NOT NULL COMMENT 'Messdatum',
            hrv_rmssd       DOUBLE           COMMENT 'Root Mean Square Successive Differences (ms)',
            hrv_sdnn        DOUBLE           COMMENT 'Standard Deviation NN Intervals (ms)',
            ppi_mean_ms     DOUBLE           COMMENT 'Mittleres Peak-to-Peak Intervall (ms)',
            hr_aus_ppi      DOUBLE           COMMENT 'Herzfrequenz abgeleitet aus PPI (bpm)',
            anzahl_samples  BIGINT           COMMENT 'Anzahl PPI-Messwerte des Tages'
        )
        USING DELTA
        COMMENT 'HRV-Metriken aus Polar ppi_*.json'
        TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true')
    """,

    'import_log': f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{schema}.import_log (
            dateiname     STRING    NOT NULL COMMENT 'Originaler Dateiname im ZIP',
            hash_md5      STRING    NOT NULL COMMENT 'MD5-Hash des Dateiinhalts',
            importiert_am TIMESTAMP          COMMENT 'Zeitpunkt des Imports',
            kategorie     STRING             COMMENT 'activity | training | heartrate | hrv | sonstige'
        )
        USING DELTA
        COMMENT 'Import-Protokoll – verhindert Doppelimport'
        TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true')
    """,
}

# ── Tabellen anlegen ─────────────────────────────────────────
for name, sql in tabellen_sql.items():
    try:
        db.abfrage(sql)
        print(f"✅ {catalog}.{schema}.{name} – bereit")
    except Exception as e:
        print(f"❌ Fehler bei '{name}': {e}")

## 5 – Erstimport: ZIP verarbeiten

ZIP-Datei in `input/` ablegen, dann diese Zelle ausführen.

In [ ]:
import glob
from pathlib import Path

# Verfügbare ZIP-Dateien anzeigen
input_ordner = Path('..') / 'input'
zip_dateien  = sorted(input_ordner.glob('*.zip'))

print(f"ZIP-Dateien in '{input_ordner}':")
if not zip_dateien:
    print("   ⚠️  Keine ZIP-Dateien gefunden!")
    print("   → Polar-Export-ZIP in den 'input/' Ordner kopieren")
    print("   → Polar App: Profil → Einstellungen → Eigene Daten exportieren")
else:
    for z in zip_dateien:
        groesse_mb = z.stat().st_size / 1024 / 1024
        print(f"   📦 {z.name}  ({groesse_mb:.1f} MB)")

In [ ]:
# ── ZIP importieren ──────────────────────────────────────────
# Diese Zelle nur ausführen wenn zip_dateien nicht leer ist!

if not zip_dateien:
    print("⏭️  Kein ZIP vorhanden – Zelle übersprungen")
else:
    from delta_updater import DeltaUpdater

    updater = DeltaUpdater()

    # Alle ZIP-Dateien verarbeiten (normalerweise nur eine)
    for zip_pfad in zip_dateien:
        print(f"\n▶ Verarbeite: {zip_pfad.name}")
        bericht = updater.verarbeite_zip(str(zip_pfad))

    print("\n🎉 Erstimport abgeschlossen!")

## 6 – Tabellenübersicht & Validierung

In [ ]:
# Übersicht aller Tabellen
df_uebersicht = db.tabellen_uebersicht()

In [ ]:
# Erste Zeilen jeder Tabelle zur Validierung
print("\n── Activity (erste 3 Zeilen) ──")
display(db.lade_activity().head(3))

print("\n── Training (erste 3 Zeilen) ──")
display(db.lade_training().head(3))

print("\n── Herzfrequenz (erste 3 Zeilen) ──")
display(db.lade_heartrate().head(3))

print("\n── HRV (erste 3 Zeilen) ──")
display(db.lade_hrv().head(3))

In [ ]:
# Import-Log prüfen
print("── Import-Log Übersicht ──")
display(db.import_log_uebersicht())

print("\n✅ Setup abgeschlossen – weiter mit 01_delta_update.ipynb")
db.schliessen()